# Unidad 3 — Reglas de Asociación

> **Inteligencia Computacional · USACH · Prof. Max Chacón**  
> Notebook de ejercicios: Apriori y FP-Growth con `mlxtend`.

**Temas cubiertos:**
1. Cálculo manual de soporte, confianza y lift sobre un dataset pequeño
2. Apriori con `mlxtend`
3. FP-Growth y comparación de tiempos
4. Visualización de reglas (scatter lift vs confianza)
5. Discretización de variables numéricas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

## 1. Cálculo manual — dataset del Prof. Chacón

Base de transacciones $\mathcal{D}$:

In [ ]:
transacciones = [
    ['a', 'b', 'e'],
    ['b', 'c', 'd'],
    ['a', 'b', 'e'],
    ['a', 'c', 'd'],
    ['a', 'b', 'c', 'd'],
]

# Soporte individual
from collections import Counter

n = len(transacciones)
items_flat = [item for t in transacciones for item in t]
conteos = Counter(items_flat)

print('Soporte individual:')
for item, cnt in sorted(conteos.items()):
    print(f'  {{{item}}}: {cnt}/{n} = {cnt/n:.2f}')

In [ ]:
def soporte(itemset, D):
    s = itemset if isinstance(itemset, set) else set(itemset)
    return sum(1 for t in D if s.issubset(t)) / len(D)

def confianza(A, B, D):
    return soporte(set(A) | set(B), D) / soporte(A, D)

def lift(A, B, D):
    return confianza(A, B, D) / soporte(B, D)

# Ejemplo: regla e => {a, b}
A = {'e'}
B = {'a', 'b'}
print(f'Regla e => {{a,b}}')
print(f'  Soporte    : {soporte(A|B, transacciones):.2f}')
print(f'  Confianza  : {confianza(A, B, transacciones):.2f}')
print(f'  Lift       : {lift(A, B, transacciones):.3f}')

## 2. Apriori con mlxtend

In [ ]:
te = TransactionEncoder()
te_array = te.fit_transform(transacciones)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

S_MIN = 0.40
t0 = time.perf_counter()
freq_items = apriori(df_trans, min_support=S_MIN, use_colnames=True)
t_apriori = time.perf_counter() - t0

print(f'Itemsets frecuentes (soporte >= {S_MIN}):')
print(freq_items.sort_values('support', ascending=False).to_string(index=False))

In [ ]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.6)
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']]
rules = rules.sort_values('lift', ascending=False)
print('Reglas de asociación:')
print(rules.round(3).to_string(index=False))

## 3. FP-Growth y comparación de tiempos

In [ ]:
t0 = time.perf_counter()
freq_fp = fpgrowth(df_trans, min_support=S_MIN, use_colnames=True)
t_fp = time.perf_counter() - t0

print(f'Apriori   : {t_apriori*1000:.2f} ms  →  {len(freq_items)} itemsets')
print(f'FP-Growth : {t_fp*1000:.2f} ms    →  {len(freq_fp)} itemsets')

# Verificar que los resultados son iguales
merged = freq_items.merge(freq_fp, on=['support', 'itemsets'], how='outer', indicator=True)
assert (merged['_merge'] == 'both').all(), 'Resultados distintos!'
print('✓ Ambos algoritmos producen los mismos itemsets frecuentes.')

## 4. Dataset más grande: supermercado sintético

In [ ]:
rng = np.random.default_rng(0)
productos = ['pan', 'leche', 'cerveza', 'pañales', 'mantequilla', 'cereal', 'jugo', 'agua', 'atún', 'arroz']

# Generar 500 transacciones con correlaciones artificiales
D_sintetico = []
for _ in range(500):
    t = set(rng.choice(productos, size=rng.integers(2, 6), replace=False))
    if 'cerveza' in t and rng.random() < 0.7:
        t.add('pañales')          # regla clásica del supermercado
    if 'pan' in t and rng.random() < 0.6:
        t.add('mantequilla')
    D_sintetico.append(list(t))

te2 = TransactionEncoder()
df_sup = pd.DataFrame(te2.fit_transform(D_sintetico), columns=te2.columns_)

freq_sup = fpgrowth(df_sup, min_support=0.10, use_colnames=True)
rules_sup = association_rules(freq_sup, metric='lift', min_threshold=1.1)
rules_sup = rules_sup.sort_values('lift', ascending=False)
print(f'Reglas con lift > 1.1: {len(rules_sup)}')
print(rules_sup[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).round(3).to_string(index=False))

## 5. Visualización de reglas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    rules_sup['support'],
    rules_sup['confidence'],
    c=rules_sup['lift'],
    cmap='YlOrRd',
    s=80,
    edgecolors='gray',
    linewidths=0.4
)
plt.colorbar(sc, ax=ax, label='Lift')
ax.set_xlabel('Soporte')
ax.set_ylabel('Confianza')
ax.set_title('Reglas de asociación — soporte vs confianza (color=lift)')
plt.tight_layout()
plt.show()

## 6. Discretización para variables numéricas

Para aplicar reglas de asociación a un dataset numérico (como Wisconsin BC) se discretizan las variables:

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import KBinsDiscretizer

data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)

# Discretizar las primeras 5 features en 3 bins de igual frecuencia
features_sel = data.feature_names[:5]
kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
X_disc = kbd.fit_transform(X_bc[features_sel])

# Construir transacciones booleanas con etiquetas
labels = ['bajo', 'medio', 'alto']
trans_bc = []
for i in range(len(X_disc)):
    items = [f'{features_sel[j]}_{labels[int(X_disc[i,j])]}' for j in range(len(features_sel))]
    # Agregar etiqueta de clase
    items.append('clase_' + data.target_names[data.target[i]])
    trans_bc.append(items)

te3 = TransactionEncoder()
df_bc_disc = pd.DataFrame(te3.fit_transform(trans_bc), columns=te3.columns_)

freq_bc = fpgrowth(df_bc_disc, min_support=0.15, use_colnames=True)
rules_bc = association_rules(freq_bc, metric='confidence', min_threshold=0.7)

# Mostrar reglas que predicen la clase
mask_clase = rules_bc['consequents'].apply(lambda c: any('clase' in x for x in c))
print('Reglas que predicen la clase (confianza ≥ 0.70):')
print(rules_bc[mask_clase][['antecedents', 'consequents', 'support', 'confidence', 'lift']]
      .sort_values('lift', ascending=False).head(8).round(3).to_string(index=False))

## Resumen

| Concepto | Valor clave |
|---|---|
| Propiedad anti-monotónica | La base de todo algoritmo eficiente |
| Apriori vs FP-Growth | Mismo resultado; FP-Growth más rápido (sin candidatos) |
| Lift > 1 | Asociación positiva (más probable que por azar) |
| Lift = 1 | Independencia estadística |
| Discretización | Obligatoria para variables continuas |

> **Ejercicio propuesto**: para el dataset del L2, aplicar igual-frecuencia con 4 bins y buscar reglas que impliquen la clase objetivo con lift > 2.